# 第4回　Python統計処理入門
## ―― 計算は機械に、判断は人間に

統計学Ⅰ（B）　／　北星学園大学

### 今日から、あなたがコードを書く

ここまで電卓やグラフ用紙でやってきた人、お疲れさま。**これからは1秒だ。**

第2回（代表値）・第3回（ばらつき）でやったことを、今日は**自分のコードで**求める。といっても全部書くわけではない。コードの大半は用意してある。あなたがやるのは、空欄 **`____`** を埋めることだけ。

> ルール：セルを上から実行し、**`____` を正しい言葉に書き換えてから** ▶ を押す。
> エラーが出ても大丈夫。エラーは「ここが違うよ」という**お知らせ**であって、失敗ではない。

In [ ]:
# 準備（このセルはそのまま実行してOK）
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan
print("準備OK")

---
## 1. データは「表」＝ DataFrame

読み込んだデータ `df` は、Excelのような**表**だ。Pythonではこれを **DataFrame** と呼ぶ。まず大きさと列名を見てみよう。（このセルはそのまま実行）

In [ ]:
print("行数・列数:", df.shape)        # (学生数, 列数)
print("列名:", list(df.columns))     # どんな列があるか
df.head()                            # 先頭5行を表示

---
## 2. 列を1つ取り出す　🔧穴埋め①

表から1列を取り出すには `df["列名"]` と書く。上のセルで見た列名から **`テスト点`** を取り出そう。

下の `____` を **テスト点** に書き換えて実行。

In [ ]:
# ____ に列名を入れる（前のセルの列名を見てね）
df["____"].head()

---
## 3. 代表値をコードで（第2回の復習）　🔧穴埋め②③

第2回で手計算した平均・中央値・最頻値を、今度はコードで。
- 平均は `.mean()`　← これは見本
- 中央値は `.median()`、最頻値は `.mode()`

`____` に正しいメソッド名を入れよう。

In [ ]:
print("平均　", df["テスト点"].mean())      # 見本
print("中央値", df["テスト点"].____())       # ヒント: median
print("最頻値", df["テスト点"].____().iloc[0]) # ヒント: mode

---
## 4. ばらつきをコードで（第3回の復習）　🔧穴埋め④

標準偏差は `.std()`、分散は `.var()`。`____` に **std** を入れて、第3回で見たSD（約11.6点）と一致するか確かめよう。

In [ ]:
print("標準偏差", df["テスト点"].____())   # ヒント: std
print("分散　　", df["テスト点"].var())

---
## 5. describe() ―― 一気に要約

`.describe()` を使うと、件数・平均・SD・最小最大・四分位数をまとめて出せる。手計算なら何十分もかかる作業が一瞬。（そのまま実行）

In [ ]:
df["テスト点"].describe()

---
## 6. グループごとに集計　🔧穴埋め⑤

「**学部ごと**のテスト点平均」を出す。`df.groupby("グループにする列")["集計する列"].mean()` と書く。
`____` に **学部** を入れよう。どの学部が一番高い？

In [ ]:
# ____ にグループにしたい列名（学部）を入れる
df.groupby("____")["テスト点"].mean()

---
## 7. 欠損を確認（そのまま実行）

第2回で仕込んだ「汚れ」（未記入）が、コードだと一瞬で見つかる。どの列にいくつ欠損があるか。

In [ ]:
df.isna().sum()

---
## 8. グラフを1行で　🔧穴埋め⑥

第2・3回で見たヒストグラムも、コードなら1行。`____` に **テスト点** を入れよう。

In [ ]:
plt.figure(figsize=(7,3.5))
plt.hist(df["____"], bins=25, color="#80cbc4", edgecolor="white")  # ____ に列名
plt.xlabel("テスト点"); plt.ylabel("人数"); plt.title("ヒストグラム（手作業ゼロ）")
plt.show()

---
## 今日のまとめ

- データ＝**表(DataFrame)**。`df["列名"]` で列を取り出す。
- 代表値 `.mean() .median() .mode()`、ばらつき `.std() .var()`、要約 `.describe()`。
- グループ集計 `df.groupby("列")["列"].mean()`、欠損 `df.isna().sum()`、グラフ `plt.hist(...)`。

第2・3回で**手で**求めた値と、今日**コードで**求めた値は同じだったはず。違いは速さと正確さ ―― **計算は機械に任せ、人間は『どの数字を信じるか』の判断に集中する。** 次回からはこの道具の上に、相関・推定・検定…を積み上げていく。

> 🔧 **困ったとき（よくあるエラー）**
> - `KeyError: '____'` → 列名のスペル違い。前のセルの `列名:` と見比べる。
> - `SyntaxError` / インデント → 半角・全角、カッコの対応、行頭の空白を確認。
> - 文字化け（□□） → 先頭の `japanize-matplotlib` のセルを実行したか確認。

**課題（Moodle）**：あなたのノートが出力した数値を書き写して答える＋「手計算とコード、役割の違い」の記述。